# Train and test set

### Feature groups
- raw Sentinel-2 bands
- indices based on raw Sentinel-2 bands (e.g. NDVI, NDWI, SAVI, EVI)

### More advanced ideas
- seasonal, quarterly and monthly time series
- min, max, median indices

## Setup

In [ ]:
import pandas as pd
from google.colab import drive

In [ ]:
drive.mount("/content/drive")

In [ ]:
%cd /content/drive/MyDrive/land_cover_classification_kaza

## Load and prepare raw data

We're using the labels provided by Nuno and the according Sentinel-2 bands.

In [ ]:
raw_data = pd.read_csv("data/raw_data.csv")
raw_data.shape

In [ ]:
raw_data

In [ ]:
raw_data = raw_data.drop(["system:index", ".geo"], axis=1)

In [ ]:
# remove land cover class deforestation as it is not needed in this use case
raw_data = raw_data[raw_data["Landcover"] != "Deforestation"]
raw_data.shape

In [ ]:
raw_data["Landcover"] = raw_data["Landcover"].str.capitalize()

## Downsample and balance raw data

In [ ]:
raw_data["Landcover"].value_counts()

In [ ]:
raw_data["LC_Nr"].value_counts()

In [ ]:
def sample_train_and_test_data(df, land_cover_class, train_fraction, desired_train_samples=None):
    random_state = 42

    # filter and shuffle df
    df = df[df["Landcover"] == land_cover_class]
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # calculate n train samples
    n_train_samples = int(round(df.shape[0] * train_fraction, 0))

    # sample train and test samples
    train_samples = df.iloc[:n_train_samples]
    test_samples = df.iloc[n_train_samples:]

    # downsample train samples if specified
    if desired_train_samples is not None:
        train_samples = train_samples.sample(
            n=desired_train_samples, random_state=random_state
        ).reset_index(drop=True)

    print(
        f"land cover class: {land_cover_class}; train samples: {train_samples.shape[0]}; test samples: {test_samples.shape[0]}"
    )

    return train_samples, test_samples

In [ ]:
train_fraction = 0.7
train_forest, test_forest = sample_train_and_test_data(
    raw_data, "Forest", train_fraction, desired_train_samples=500
)
train_cropland, test_cropland = sample_train_and_test_data(
    raw_data, "Cropland", train_fraction, desired_train_samples=500
)
train_wetland, test_wetland = sample_train_and_test_data(
    raw_data, "Wetland", train_fraction, desired_train_samples=500
)
train_shrub, test_shrub = sample_train_and_test_data(
    raw_data, "Shrub", train_fraction, desired_train_samples=500
)
train_grass, test_grass = sample_train_and_test_data(
    raw_data, "Grass", train_fraction, desired_train_samples=500
)
train_bare, test_bare = sample_train_and_test_data(raw_data, "Bare", train_fraction)
train_water, test_water = sample_train_and_test_data(raw_data, "Water", train_fraction)
train_built_up, test_built_up = sample_train_and_test_data(raw_data, "Built up", train_fraction)

In [ ]:
train = pd.concat(
    [
        train_forest,
        train_cropland,
        train_wetland,
        train_shrub,
        train_grass,
        train_bare,
        train_water,
        train_built_up,
    ],
    ignore_index=True,
)
train

In [ ]:
test = pd.concat(
    [
        test_forest,
        test_cropland,
        test_wetland,
        test_shrub,
        test_grass,
        test_bare,
        test_water,
        test_built_up,
    ],
    ignore_index=True,
)
test

## Compute indices

In [ ]:
def ndvi(df, nir_band, red_band):
    ndvi = (df[nir_band] - df[red_band]) / (df[nir_band] + df[red_band])
    return ndvi

In [ ]:
def ndwi_mcf(df, green_band, nir_band):
    ndwi_mcf = (df[green_band] - df[nir_band]) / (df[green_band] + df[nir_band])
    return ndwi_mcf

In [ ]:
def ndwi_gao(df, nir_band, swir_band):
    ndwi_gao = (df[nir_band] - df[swir_band]) / (df[nir_band] + df[swir_band])
    return ndwi_gao

In [ ]:
def savi(df, nir_band, red_band):
    savi = (df[nir_band] - df[red_band]) / (df[nir_band] + df[red_band] + 0.5) * (1.0 + 0.5)
    return savi

In [ ]:
def evi(df, nir_band, red_band, blue_band):
    evi = 2.5 * (
        (df[nir_band] - df[red_band]) / (df[nir_band] + 6 * df[red_band] - 7.5 * df[blue_band] + 1)
    )
    return evi

In [ ]:
add_indices = False

In [ ]:
if add_indices:
    train["NDVI_Q1"] = ndvi(train, "B8_Q1", "B4_Q1")
    train["NDWI_MCF_Q1"] = ndwi_mcf(train, "B3_Q1", "B8_Q1")
    train["NDWI_GAO_Q1"] = ndwi_gao(train, "B8_Q1", "B12_Q1")
    train["SAVI_Q1"] = savi(train, "B8_Q1", "B4_Q1")
    train["EVI_Q1"] = evi(train, "B8_Q1", "B4_Q1", "B2_Q1")
    train["NDVI_Q2"] = ndvi(train, "B8_Q2", "B4_Q2")
    train["NDWI_MCF_Q2"] = ndwi_mcf(train, "B3_Q2", "B8_Q2")
    train["NDWI_GAO_Q2"] = ndwi_gao(train, "B8_Q2", "B12_Q2")
    train["SAVI_Q2"] = savi(train, "B8_Q2", "B4_Q2")
    train["EVI_Q2"] = evi(train, "B8_Q2", "B4_Q2", "B2_Q2")
    train["NDVI_Q3"] = ndvi(train, "B8_Q3", "B4_Q3")
    train["NDWI_MCF_Q3"] = ndwi_mcf(train, "B3_Q3", "B8_Q3")
    train["NDWI_GAO_Q3"] = ndwi_gao(train, "B8_Q3", "B12_Q3")
    train["SAVI_Q3"] = savi(train, "B8_Q3", "B4_Q3")
    train["EVI_Q3"] = evi(train, "B8_Q3", "B4_Q3", "B2_Q3")
    train["NDVI_Q4"] = ndvi(train, "B8_Q4", "B4_Q4")
    train["NDWI_MCF_Q4"] = ndwi_mcf(train, "B3_Q4", "B8_Q4")
    train["NDWI_GAO_Q4"] = ndwi_gao(train, "B8_Q4", "B12_Q4")
    train["SAVI_Q4"] = savi(train, "B8_Q4", "B4_Q4")
    train["EVI_Q4"] = evi(train, "B8_Q4", "B4_Q4", "B2_Q4")
    train

In [ ]:
if add_indices:
    test["NDVI_Q1"] = ndvi(test, "B8_Q1", "B4_Q1")
    test["NDWI_MCF_Q1"] = ndwi_mcf(test, "B3_Q1", "B8_Q1")
    test["NDWI_GAO_Q1"] = ndwi_gao(test, "B8_Q1", "B12_Q1")
    test["SAVI_Q1"] = savi(test, "B8_Q1", "B4_Q1")
    test["EVI_Q1"] = evi(test, "B8_Q1", "B4_Q1", "B2_Q1")
    test["NDVI_Q2"] = ndvi(test, "B8_Q2", "B4_Q2")
    test["NDWI_MCF_Q2"] = ndwi_mcf(test, "B3_Q2", "B8_Q2")
    test["NDWI_GAO_Q2"] = ndwi_gao(test, "B8_Q2", "B12_Q2")
    test["SAVI_Q2"] = savi(test, "B8_Q2", "B4_Q2")
    test["EVI_Q2"] = evi(test, "B8_Q2", "B4_Q2", "B2_Q2")
    test["NDVI_Q3"] = ndvi(test, "B8_Q3", "B4_Q3")
    test["NDWI_MCF_Q3"] = ndwi_mcf(test, "B3_Q3", "B8_Q3")
    test["NDWI_GAO_Q3"] = ndwi_gao(test, "B8_Q3", "B12_Q3")
    test["SAVI_Q3"] = savi(test, "B8_Q3", "B4_Q3")
    test["EVI_Q3"] = evi(test, "B8_Q3", "B4_Q3", "B2_Q3")
    test["NDVI_Q4"] = ndvi(test, "B8_Q4", "B4_Q4")
    test["NDWI_MCF_Q4"] = ndwi_mcf(test, "B3_Q4", "B8_Q4")
    test["NDWI_GAO_Q4"] = ndwi_gao(test, "B8_Q4", "B12_Q4")
    test["SAVI_Q4"] = savi(test, "B8_Q4", "B4_Q4")
    test["EVI_Q4"] = evi(test, "B8_Q4", "B4_Q4", "B2_Q4")
    test

## Save train and test set

In [ ]:
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)